# 04 — Cohortes et deux scores

Cohorte ML : `ModeSortie == Domicile`.

Horizon télé **principal = 7 jours**, fenêtre **fixe** identique pour tous (anti-fuite : on ne tronque pas à la réadmission).

J+14 = analyse de sensibilité (couverture capteurs).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
from src.data.paths import ProjectPaths, get_project_root
from src.data.load import load_csv
from src.data.quality import recalculate_duree_sejour, filter_objets_connectes_bons
from src.data.cohort import filter_domicile, patient_level_split
import pandas as pd

paths = ProjectPaths(root=get_project_root())
sejours = recalculate_duree_sejour(load_csv(paths, "sejours.csv", from_raw=False))
print(sejours["ModeSortie"].value_counts())
el = patient_level_split(filter_domicile(sejours))
print(el["split"].value_counts())
print("prévalence", float(el["Readmission30j"].mean()))

ModeSortie
Domicile     480
Transfert    144
Deces         92
EHPAD         87
HAD           75
Name: count, dtype: int64
split
train    340
test      72
val       68
Name: count, dtype: int64
prévalence 0.18541666666666667


In [3]:
oc = filter_objets_connectes_bons(load_csv(paths, "objets_connectes.csv", from_raw=False))
oc["Horodatage"] = pd.to_datetime(oc["Horodatage"], errors="coerce")
el = el.copy()
el["DateSortie"] = pd.to_datetime(el["DateSortie"], errors="coerce")

def coverage(horizon):
    n = 0
    for _, r in el.iterrows():
        fin = r["DateSortie"] + pd.Timedelta(int(horizon), unit="D")
        sub = oc[(oc["PatientID"]==r["PatientID"]) & (oc["Horodatage"]>r["DateSortie"]) & (oc["Horodatage"]<=fin)]
        n += int(len(sub)>0)
    return n

print("séjours avec ≥1 mesure Bon J+7", coverage(7), "/", len(el))
print("séjours avec ≥1 mesure Bon J+14", coverage(14), "/", len(el))

séjours avec ≥1 mesure Bon J+7 10 / 480


séjours avec ≥1 mesure Bon J+14 12 / 480


In [4]:
assert (el.groupby("PatientID")["split"].nunique()==1).all()
print("OK cohortes")

OK cohortes
